# Veri On Isleme Pipeline
## Eksik Deger Analizi | Aykiri Deger Analizi | KNN Imputation | Encoding

**Veri Seti:** `train.csv` - Uyku & Bilissel Performans Datathon Veri Seti

## 1. Kutuphanelerin Yuklenmesi

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

## 2. Veri Setinin Yuklenmesi ve Genel Bilgi

In [ ]:
df = pd.read_csv('train.csv')
print(f'Satir sayisi: {df.shape[0]}')
print(f'Sutun sayisi: {df.shape[1]}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

## 3. Eksik Deger Analizi

In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Eksik Sayi': missing,
    'Eksik Yuzde (%)': missing_pct,
    'Veri Tipi': df.dtypes
}).sort_values('Eksik Sayi', ascending=False)

print(f'Toplam eksik hucre: {df.isnull().sum().sum()}')
print(f'Toplam hucre: {df.shape[0] * df.shape[1]}')
print(f'Genel eksiklik orani: {(df.isnull().sum().sum() / (df.shape[0]*df.shape[1]) * 100):.2f}%')
missing_df

### 3.1 Eksik Deger Gorsellestirme

In [ ]:
missing_cols = missing_df[missing_df['Eksik Sayi'] > 0]
cols_with_missing = missing_cols.index.tolist()
vals = missing_cols['Eksik Sayi'].values
pcts = missing_cols['Eksik Yuzde (%)'].values

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
colors = sns.color_palette('Reds_r', len(cols_with_missing))

bars = axes[0].barh(cols_with_missing, vals, color=colors)
axes[0].set_title('Eksik Deger Sayilari', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Eksik Sayi')
for bar, v in zip(bars, vals):
    axes[0].text(bar.get_width() + max(vals)*0.01, bar.get_y() + bar.get_height()/2, f'{int(v)}', va='center')

bars2 = axes[1].barh(cols_with_missing, pcts, color=colors)
axes[1].set_title('Eksik Deger Yuzdeleri (%)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Yuzde (%)')
for bar, v in zip(bars2, pcts):
    axes[1].text(bar.get_width() + max(pcts)*0.01, bar.get_y() + bar.get_height()/2, f'{v:.1f}%', va='center')

plt.tight_layout()
plt.show()

### 3.2 Eksik Deger Deseni (Heatmap)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
sample_idx = np.random.RandomState(42).choice(len(df), 1000, replace=False)
sns.heatmap(df.iloc[sample_idx][cols_with_missing].isnull().astype(int),
            cbar_kws={'label': 'Eksik (1) / Mevcut (0)'},
            cmap='YlOrRd', ax=ax, yticklabels=False)
ax.set_title('Eksik Deger Deseni (1000 Ornek)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.3 Semantik/Ortuk Eksik Deger Kontrolu

In [ ]:
suspect_values = ['?', 'NA', 'N/A', 'null', 'None', '-', '', ' ', 'Bilinmiyor']
cat_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
for col in cat_cols:
    unique_vals = df[col].unique()
    found = [v for v in suspect_values if v in unique_vals]
    if found:
        print(f"  UYARI '{col}': Supheli degerler -> {found}")
    print(f"  '{col}' benzersiz degerler ({df[col].nunique()}): {list(unique_vals[:10])}")

## 4. Aykiri Deger Analizi (IQR Yontemi)

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'id' in num_cols:
    num_cols.remove('id')

outlier_summary = []
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    count = len(outliers)
    pct = count / len(df) * 100
    outlier_summary.append({
        'Sutun': col, 'Q1': round(Q1, 2), 'Q3': round(Q3, 2), 'IQR': round(IQR, 2),
        'Alt Sinir': round(lower, 2), 'Ust Sinir': round(upper, 2),
        'Aykiri Sayi': count, 'Aykiri Yuzde (%)': round(pct, 2)
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values('Aykiri Sayi', ascending=False)
outlier_df

### 4.1 Aykiri Deger Oranlari

In [ ]:
has_outliers = outlier_df[outlier_df['Aykiri Sayi'] > 0]
fig, ax = plt.subplots(figsize=(14, 7))
colors = sns.color_palette('OrRd_r', len(has_outliers))
bars = ax.barh(has_outliers['Sutun'], has_outliers['Aykiri Yuzde (%)'], color=colors)
ax.set_title('Aykiri Deger Oranlari (IQR Yontemi)', fontsize=14, fontweight='bold')
ax.set_xlabel('Aykiri Deger Yuzdesi (%)')
for bar, v in zip(bars, has_outliers['Aykiri Yuzde (%)'].values):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, f'{v:.1f}%', va='center')
plt.tight_layout()
plt.show()

### 4.2 Boxplot Grafikleri

In [ ]:
n = len(num_cols)
ncols_plot = 3
nrows_plot = (n + ncols_plot - 1) // ncols_plot
fig, axes = plt.subplots(nrows_plot, ncols_plot, figsize=(18, nrows_plot * 4))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.boxplot(data=df, y=col, ax=axes[i], color='#4ECDC4',
                flierprops=dict(marker='o', markersize=3, alpha=0.3))
    axes[i].set_title(col, fontsize=11, fontweight='bold')
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Sayisal Degiskenlerin Boxplot Grafikleri', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 4.3 Histogram + KDE Dagilimlari

In [ ]:
fig, axes = plt.subplots(nrows_plot, ncols_plot, figsize=(18, nrows_plot * 4))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i], color='#3498db', alpha=0.7, bins=50)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Ort: {df[col].mean():.2f}')
    axes[i].axvline(df[col].median(), color='green', linestyle='--', label=f'Med: {df[col].median():.2f}')
    axes[i].legend(fontsize=8)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Sayisal Degisken Dagilimlari (Histogram + KDE)', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 4.4 Carpiklik ve Basiklik Istatistikleri

In [ ]:
desc = df[num_cols].describe().T
desc['skewness'] = df[num_cols].skew()
desc['kurtosis'] = df[num_cols].kurtosis()
desc.round(3)

## 5. Eksik Deger Doldurma (KNN Imputation)

**Strateji:**
- `ruh_sagligi_durumu` ve `kronotip` -> **Mod** (en sik tekrar eden deger)
- `meslek` -> **KNN Imputation** (k=5, distance weighted)
- `vucut_kitle_indeksi`, `stres_skoru`, `uyku_oncesi_kafein_mg` -> **KNN Imputation**

### 5.1 Kategorik: ruh_sagligi_durumu & kronotip -> Mod

In [ ]:
for col in ['ruh_sagligi_durumu', 'kronotip']:
    df[col] = df[col].astype(object)
    mode_val = df[col].mode()[0]
    eksik = df[col].isnull().sum()
    df[col] = df[col].fillna(mode_val)
    print(f"{col}: {eksik} eksik deger '{mode_val}' ile dolduruldu. Kalan: {df[col].isnull().sum()}")

### 5.2 Kategorik: meslek -> KNN Imputation

In [ ]:
meslek_eksik = df['meslek'].isnull().sum()
df['meslek'] = df['meslek'].astype(object)
print(f'meslek: {meslek_eksik} eksik deger KNN ile doldurulacak.')

cat_cols_for_knn = ['cinsiyet', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi', 'meslek']

# Gecici Label Encoding (KNN icin)
label_encoders = {}
df_encoded = df.copy()
for col in cat_cols_for_knn:
    df_encoded[col] = df_encoded[col].astype(object)
    df_encoded.loc[df[col].isnull(), col] = np.nan

for col in cat_cols_for_knn:
    le = LabelEncoder()
    mask = df_encoded[col].isnull()
    non_null_vals = df_encoded.loc[~mask, col].values
    encoded_vals = le.fit_transform(non_null_vals)
    df_encoded.loc[~mask, col] = encoded_vals.astype(float)
    df_encoded.loc[mask, col] = np.nan
    df_encoded[col] = df_encoded[col].astype(float)
    label_encoders[col] = le

num_cols_all = ['yas', 'vucut_kitle_indeksi', 'rem_yuzdesi', 'derin_uyku_yuzdesi',
            'uykuya_dalma_suresi_dk', 'gecelik_uyanma_sayisi', 'uyku_oncesi_kafein_mg',
            'uyku_oncesi_ekran_suresi_dk', 'gunluk_adim_sayisi', 'sekerleme_suresi_dk',
            'stres_skoru', 'gunluk_calisma_saati', 'dinlenik_nabiz_bpm',
            'oda_sicakligi_celsius', 'hafta_sonu_uyku_farki_saat', 'bilissel_performans_skoru']

features_for_meslek = cat_cols_for_knn + num_cols_all

# Scaling + KNN
scaler_meslek = StandardScaler()
df_mf = df_encoded[features_for_meslek].copy()
scaler_meslek.fit(df_mf.dropna())
df_scaled = pd.DataFrame(scaler_meslek.transform(df_mf), columns=features_for_meslek, index=df_mf.index)

knn = KNNImputer(n_neighbors=5, weights='distance')
print('KNN Imputer calistiriliyor...')
imputed = knn.fit_transform(df_scaled)
imputed_unscaled = pd.DataFrame(scaler_meslek.inverse_transform(imputed), columns=features_for_meslek, index=df.index)

# Decode meslek
meslek_enc = imputed_unscaled['meslek'].round().astype(int).clip(0, len(label_encoders['meslek'].classes_) - 1)
meslek_decoded = label_encoders['meslek'].inverse_transform(meslek_enc)
meslek_null_mask = df['meslek'].isnull()
df.loc[meslek_null_mask, 'meslek'] = meslek_decoded[meslek_null_mask]

print(f'meslek: {meslek_eksik} eksik deger KNN ile dolduruldu.')
print('\nDoldurulan deger dagilimi:')
print(df.loc[meslek_null_mask, 'meslek'].value_counts())

### 5.3 Numerik Degiskenler -> KNN Imputation

In [ ]:
numerik_eksik_cols = ['vucut_kitle_indeksi', 'stres_skoru', 'uyku_oncesi_kafein_mg']
for col in numerik_eksik_cols:
    print(f'{col}: {df[col].isnull().sum()} eksik deger')

df_num_knn = df.copy()
cat_all = ['cinsiyet', 'meslek', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']
le_dict = {}
for col in cat_all:
    le = LabelEncoder()
    df_num_knn[col] = le.fit_transform(df_num_knn[col])
    le_dict[col] = le

knn_features = num_cols_all + cat_all
scaler_num = StandardScaler()
df_knn_input = df_num_knn[knn_features].copy().astype(float)
scaler_num.fit(df_knn_input.dropna())
df_knn_scaled = pd.DataFrame(scaler_num.transform(df_knn_input), columns=knn_features, index=df_knn_input.index)

knn_num = KNNImputer(n_neighbors=5, weights='distance')
print('KNN Imputer calistiriliyor...')
imputed_num = knn_num.fit_transform(df_knn_scaled)
imputed_num_unscaled = pd.DataFrame(scaler_num.inverse_transform(imputed_num), columns=knn_features, index=df.index)

for col in numerik_eksik_cols:
    null_mask = df[col].isnull()
    df.loc[null_mask, col] = imputed_num_unscaled.loc[null_mask, col]
    print(f'{col}: KNN ile dolduruldu. (Doldurulan ort: {df.loc[null_mask, col].mean():.2f})')

### 5.4 Dogrulama

In [ ]:
remaining = df.isnull().sum()
print('Eksik deger kalmadi!' if remaining.sum() == 0 else remaining[remaining > 0])
print(f'Toplam kalan eksik: {remaining.sum()}')

## 6. Kategorik Degisken Encoding

**Strateji:**
- **Binary** (`cinsiyet`, `mevsim`, `gun_tipi`) -> Label Encoding (0/1)
- **Ordinal** (`kronotip`, `ruh_sagligi_durumu`) -> Ordinal Encoding
- **Nominal** (`meslek`, `ulke`) -> One-Hot Encoding

### 6.1 Binary Kategorikler -> Label Encoding

In [ ]:
binary_mappings = {
    'cinsiyet': {'Erkek': 0, 'Kadin': 1},
    'mevsim': {'Sonbahar-Kis': 0, 'Ilkbahar-Yaz': 1},
    'gun_tipi': {'Hafta ici': 0, 'Hafta sonu': 1},
}
for col, mapping in binary_mappings.items():
    df[col] = df[col].map(mapping)
    print(f'{col}: {mapping}')

### 6.2 Ordinal Kategorikler -> Ordinal Encoding

In [ ]:
kronotip_mapping = {'Gece insani': 0, 'Notr': 1, 'Sabah insani': 2}
df['kronotip'] = df['kronotip'].map(kronotip_mapping)
print(f'kronotip: {kronotip_mapping}')

ruh_mapping = {
    'Anksiyete ve depresyon': 0,
    'Depresyon': 1,
    'Anksiyete': 2,
    'Saglikli': 3
}
df['ruh_sagligi_durumu'] = df['ruh_sagligi_durumu'].map(ruh_mapping)
print(f'ruh_sagligi_durumu: {ruh_mapping}')

### 6.3 Nominal Kategorikler -> One-Hot Encoding

In [ ]:
print(f"meslek: {df['meslek'].nunique()} sinif -> One-Hot")
df = pd.get_dummies(df, columns=['meslek'], prefix='meslek', dtype=int)

print(f"ulke: {df['ulke'].nunique()} sinif -> One-Hot")
df = pd.get_dummies(df, columns=['ulke'], prefix='ulke', dtype=int)

### 6.4 Dogrulama

In [ ]:
str_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
if str_cols:
    print(f'UYARI: Hala string sutunlar var: {str_cols}')
else:
    print('Tum sutunlar sayisal formata donusturuldu.')

print(f'\nSonuc boyutu: {df.shape}')
print(f'Eksik deger: {df.isnull().sum().sum()}')

print(f'\nSutun listesi:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col} ({df[col].dtype})')

## 7. Temizlenmis Verinin Kaydedilmesi

In [ ]:
df.to_csv('train_encoded.csv', index=False)
print(f'Encoded veri kaydedildi: train_encoded.csv')
print(f'Son boyut: {df.shape}')
df.head()